In [49]:
import pyspark
import pandas
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import col,when,current_date,lit

In [41]:
spark = SparkSession.builder.appName("scd").getOrCreate()

In [42]:
# Target Table (existing dimension)

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("start_date", StringType(), True),   # or DateType()
    StructField("end_date", StringType(), True),     # None column → must define type
    StructField("is_active", IntegerType(), True)
])

target_data = [
    (1001, "Argha", "Kolkata", "2023-01-01", None, 1),
    (1002, "Rahul", "Delhi", "2023-01-01", None, 1),
    (1003, "John", "Mumbai", "2023-01-05", None, 1),
    (1004, "Sham", "Chennai", "2023-01-08", None, 1)
]

dim_df = spark.createDataFrame(target_data, schema)

dim_df.show()

+-----------+-----+-------+----------+--------+---------+
|customer_id| name|   city|start_date|end_date|is_active|
+-----------+-----+-------+----------+--------+---------+
|       1001|Argha|Kolkata|2023-01-01|    null|        1|
|       1002|Rahul|  Delhi|2023-01-01|    null|        1|
|       1003| John| Mumbai|2023-01-05|    null|        1|
|       1004| Sham|Chennai|2023-01-08|    null|        1|
+-----------+-----+-------+----------+--------+---------+



In [43]:
# Source Data (new incoming data)
define_schema = StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("city",StringType(),True)
])

source_data = [
    (1001, "Argha", "Mumbai"),  # changed city
    (1002, "Rahul", "Delhi"),   # no change
    (1003, "John", "pune"),     # changed city
    (1005, "Amit", "Pune")      # new record
]

src_df = spark.createDataFrame(source_data, define_schema)
src_df.show()

+-----------+-----+------+
|customer_id| name|  city|
+-----------+-----+------+
|       1001|Argha|Mumbai|
|       1002|Rahul| Delhi|
|       1003| John|  pune|
|       1005| Amit|  Pune|
+-----------+-----+------+



In [46]:
# Step 1: Filter Active Records
active_dim_df = dim_df.filter(col("is_active") == 1)
active_dim_df.show()

+-----------+-----+-------+----------+--------+---------+
|customer_id| name|   city|start_date|end_date|is_active|
+-----------+-----+-------+----------+--------+---------+
|       1001|Argha|Kolkata|2023-01-01|    null|        1|
|       1002|Rahul|  Delhi|2023-01-01|    null|        1|
|       1003| John| Mumbai|2023-01-05|    null|        1|
|       1004| Sham|Chennai|2023-01-08|    null|        1|
+-----------+-----+-------+----------+--------+---------+



In [47]:
# Step 2: Join
join_df = src_df.alias("src").join(
    active_dim_df.alias("dim"),
    "customer_id",
    "left"
)
join_df.show()

+-----------+-----+------+-----+-------+----------+--------+---------+
|customer_id| name|  city| name|   city|start_date|end_date|is_active|
+-----------+-----+------+-----+-------+----------+--------+---------+
|       1001|Argha|Mumbai|Argha|Kolkata|2023-01-01|    null|        1|
|       1002|Rahul| Delhi|Rahul|  Delhi|2023-01-01|    null|        1|
|       1003| John|  pune| John| Mumbai|2023-01-05|    null|        1|
|       1005| Amit|  Pune| null|   null|      null|    null|     null|
+-----------+-----+------+-----+-------+----------+--------+---------+



In [48]:
# Step 3: Detect Changes
changed_df = join_df.filter(
    (col("dim.customer_id").isNotNull()) &
    (
        (col("src.name") != col("dim.name")) |
        (col("src.city") != col("dim.city"))
    )
)
changed_df.show()

+-----------+-----+------+-----+-------+----------+--------+---------+
|customer_id| name|  city| name|   city|start_date|end_date|is_active|
+-----------+-----+------+-----+-------+----------+--------+---------+
|       1001|Argha|Mumbai|Argha|Kolkata|2023-01-01|    null|        1|
|       1003| John|  pune| John| Mumbai|2023-01-05|    null|        1|
+-----------+-----+------+-----+-------+----------+--------+---------+



In [51]:
# Step 4: Expire Old Records
expired_df = changed_df.select(
    col("customer_id"),
    col("dim.name"),
    col("dim.city"),
    col("dim.start_date"),
    current_date().alias("end_date"),
    lit(0).alias("is_active")   # 👈 changed
)
expired_df.show()

+-----------+-----+-------+----------+----------+---------+
|customer_id| name|   city|start_date|  end_date|is_active|
+-----------+-----+-------+----------+----------+---------+
|       1001|Argha|Kolkata|2023-01-01|2026-03-21|        0|
|       1003| John| Mumbai|2023-01-05|2026-03-21|        0|
+-----------+-----+-------+----------+----------+---------+



In [52]:
# Step 5: Insert New Version
new_version_df = changed_df.select(
    col("customer_id"),
    col("src.name"),
    col("src.city"),
    current_date().alias("start_date"),
    lit(None).cast("date").alias("end_date"),
    lit(1).alias("is_active")   # 👈 active
)
new_version_df.show()

+-----------+-----+------+----------+--------+---------+
|customer_id| name|  city|start_date|end_date|is_active|
+-----------+-----+------+----------+--------+---------+
|       1001|Argha|Mumbai|2026-03-21|    null|        1|
|       1003| John|  pune|2026-03-21|    null|        1|
+-----------+-----+------+----------+--------+---------+



In [53]:
# Step 6: New Customers
new_customer_df = join_df.filter(
    col("dim.customer_id").isNull()
).select(
    col("customer_id"),
    col("src.name"),
    col("src.city"),
    current_date().alias("start_date"),
    lit(None).cast("date").alias("end_date"),
    lit(1).alias("is_active")
)
new_customer_df.show()

+-----------+----+----+----------+--------+---------+
|customer_id|name|city|start_date|end_date|is_active|
+-----------+----+----+----------+--------+---------+
|       1005|Amit|Pune|2026-03-21|    null|        1|
+-----------+----+----+----------+--------+---------+



In [55]:
# Step 7: Unchanged + Missing Records
unchanged_df = dim_df.join(
    changed_df.select("customer_id"),
    "customer_id",
    "left_anti"
)
unchanged_df.show()

+-----------+-----+-------+----------+--------+---------+
|customer_id| name|   city|start_date|end_date|is_active|
+-----------+-----+-------+----------+--------+---------+
|       1002|Rahul|  Delhi|2023-01-01|    null|        1|
|       1004| Sham|Chennai|2023-01-08|    null|        1|
+-----------+-----+-------+----------+--------+---------+



In [56]:
# Step 8: Final Data
final_df = unchanged_df.unionByName(expired_df) \
                      .unionByName(new_version_df) \
                      .unionByName(new_customer_df)
final_df.show()

+-----------+-----+-------+----------+----------+---------+
|customer_id| name|   city|start_date|  end_date|is_active|
+-----------+-----+-------+----------+----------+---------+
|       1002|Rahul|  Delhi|2023-01-01|      null|        1|
|       1004| Sham|Chennai|2023-01-08|      null|        1|
|       1001|Argha|Kolkata|2023-01-01|2026-03-21|        0|
|       1003| John| Mumbai|2023-01-05|2026-03-21|        0|
|       1001|Argha| Mumbai|2026-03-21|      null|        1|
|       1003| John|   pune|2026-03-21|      null|        1|
|       1005| Amit|   Pune|2026-03-21|      null|        1|
+-----------+-----+-------+----------+----------+---------+



In [57]:
sql_df = spark.sql(
'''
MERGE INTO dim_customer AS target
USING source_customer AS source
ON target.customer_id = source.customer_id AND target.is_active = 1

WHEN MATCHED AND (
    target.name <> source.name OR
    target.city <> source.city
)
THEN UPDATE SET
    target.end_date = current_date(),
    target.is_active = 0

WHEN NOT MATCHED
THEN INSERT (
    customer_id, name, city, start_date, end_date, is_active
)
VALUES (
    source.customer_id, source.name, source.city,
    current_date(), NULL, 1
)
'''
)
sql_df.show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `dim_customer` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 11;
'MergeIntoTable (('target.customer_id = 'source.customer_id) AND ('target.is_active = 1)), [updateaction(Some((NOT ('target.name = 'source.name) OR NOT ('target.city = 'source.city))), assignment('target.end_date, current_date(Some(Asia/Calcutta))), assignment('target.is_active, 0))], [insertaction(None, assignment('customer_id, 'source.customer_id), assignment('name, 'source.name), assignment('city, 'source.city), assignment('start_date, current_date(Some(Asia/Calcutta))), assignment('end_date, null), assignment('is_active, 1))]
:- 'SubqueryAlias target
:  +- 'UnresolvedRelation [dim_customer], [], 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50311)
Traceback (most recent call last):
  File "D:\Python\Python310\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "D:\Python\Python310\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "D:\Python\Python310\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "D:\Python\Python310\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "d:\ADF_DB\.venv\lib\site-packages\pyspark\accumulators.py", line 281, in handle
    poll(accum_updates)
  File "d:\ADF_DB\.venv\lib\site-packages\pyspark\accumulators.py", line 253, in poll
    if func():
  File "d:\ADF_DB\.venv\lib\site-packages\pyspark\accumulators.py", line 257, in accum_updates
    num_updates = read_int(self.rfile)
  File "d: